# Question 13 - 238. Product of Array Except Self

Given an integer array `nums`, return an array `answer` such that `answer[i]` is equal to the product of all the elements of `nums` except `nums[i]`.

The product of any prefix or suffix of `nums` is **guaranteed** to fit in a **32-bit** integer.

You must write an algorithm that runs in `O(n)` time and **without using the division operation**.

**Example 1:**

    Input: nums = [1,2,3,4]
    Output: [24,12,8,6]

**Example 2:**

    Input: nums = [-1,1,0,-3,3]
    Output: [0,0,9,0,0]

**Constraints:**

- `2 <= nums.length <= 10^5`
- `-30 <= nums[i] <= 30`
- The product of any prefix or suffix of `nums` is **guaranteed** to fit in a **32-bit** integer.

**Follow up:** Can you solve the problem in `O(1)` extra space complexity? (The output array does **not** count as extra space for space complexity analysis.)

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Prefix / suffix products (two passes, reuse the output array)

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force | O(n²) | O(1) extra |
| 1½ Prefix + suffix arrays | O(n) | O(n) extra |
| 2️⃣ Optimized | O(n) | O(1) extra (output not counted) |

> 🗣️ lines are an **interview script**: roughly what to say out loud at each step.

### Step 0 · Clarify before coding

🗣️ *"For each index I need the product of everything else. The obvious trick is total product divided by `nums[i]`, but division is banned, and it would break on zeros anyway."*

- Division not allowed → **confirmed**. (Even if it were allowed, a zero in the array makes `total / 0` undefined, so you'd need special cases for one zero vs. two zeros.)
- Zeros and negatives allowed? → **Yes.**
- Overflow? → Guaranteed to fit in 32-bit (and Python ints don't overflow anyway).
- Does the output array count as extra space? → **No.** That's what makes the O(1) follow-up possible.

### Step 1 · Brute force

**Idea:** For each `i`, multiply every `nums[j]` with `j != i`.

🗣️ *"The direct way is a nested loop: for each index, multiply all the others. That's O(n²), about 10¹⁰ operations for n = 10⁵, which is far too slow. The waste is that the products for neighbouring indices share almost all of their factors."*

In [1]:
class SolutionBrute:
    def productExceptSelf(self, nums: list[int]) -> list[int]:
        n = len(nums)
        answer = []
        for i in range(n):
            p = 1
            for j in range(n):
                if j != i:
                    p *= nums[j]
            answer.append(p)
        return answer

### Step 2 · Optimize

**Bottleneck:** `answer[i]` and `answer[i+1]` share n − 2 factors, but brute force recomputes them from scratch.

**Key insight:** split "everything except `i`" into two halves:

```
answer[i] = (nums[0] * … * nums[i-1])  *  (nums[i+1] * … * nums[n-1])
            └──── prefix product ────┘     └──── suffix product ────┘
```

Both halves can be built incrementally. `prefix[i+1] = prefix[i] * nums[i]`, and likewise for the suffix from the right. That's O(1) per index.

**Version A (clear, O(n) extra):** build a `prefix` array and a `suffix` array, then multiply them.

**Version B (follow-up, O(1) extra):**
1. **Left pass:** write prefix products straight into `answer`.
2. **Right pass:** walk from the right with a single running variable `suffix`, and multiply it into `answer[i]`.

🗣️ *"Each answer is the product of everything to its left times everything to its right. I can build the left products in one pass and the right products in another. To get O(1) extra space, I store the left products in the output array and keep the right product as a single running variable on the way back."*

#### Choosing the data structure & algorithm: why this, why not that

| Option | Verdict | Why / why not |
|---|---|---|
| Nested loops | ❌ | O(n²). It recomputes shared factors. |
| Total product ÷ `nums[i]` | ❌ | **Forbidden**, and it breaks on zeros (needs special cases for 1 zero / 2+ zeros). |
| Prefix array + suffix array | ✅ (clear) | O(n) time, easy to explain, but two extra O(n) arrays. |
| **Prefix in output + running suffix scalar** | ✅ (best) | Same O(n) time, and O(1) extra space because the output array does double duty. |
| Log-sum trick (`exp(Σlog)`) | ❌ | Floating-point error, and it can't handle zeros or negatives. |

**Why zeros "just work":** we never divide. A zero simply makes every prefix or suffix that includes it equal to 0, which is the correct result.

In [2]:
class SolutionPrefixSuffix:
    def productExceptSelf(self, nums: list[int]) -> list[int]:
        n = len(nums)
        prefix = [1] * n      # prefix[i] = product of nums[0..i-1]
        suffix = [1] * n      # suffix[i] = product of nums[i+1..n-1]
        for i in range(1, n):
            prefix[i] = prefix[i - 1] * nums[i - 1]
        for i in range(n - 2, -1, -1):
            suffix[i] = suffix[i + 1] * nums[i + 1]
        return [prefix[i] * suffix[i] for i in range(n)]


class Solution:
    def productExceptSelf(self, nums: list[int]) -> list[int]:
        n = len(nums)
        answer = [1] * n
        # left pass: answer[i] = product of everything left of i
        for i in range(1, n):
            answer[i] = answer[i - 1] * nums[i - 1]
        # right pass: multiply in product of everything right of i
        suffix = 1
        for i in range(n - 1, -1, -1):
            answer[i] *= suffix
            suffix *= nums[i]
        return answer

### Step 3 · Toy example walkthrough

`nums = [1, 2, 3, 4]`

**Left pass** (`answer[i]` = product of everything left of i):

| i | answer[i] = answer[i−1] × nums[i−1] | answer |
|---|---|---|
| 0 | 1 (nothing on the left) | [1, 1, 1, 1] |
| 1 | 1 × 1 = 1 | [1, 1, 1, 1] |
| 2 | 1 × 2 = 2 | [1, 1, 2, 1] |
| 3 | 2 × 3 = 6 | [1, 1, 2, 6] |

**Right pass** (`suffix` = product of everything right of i):

| i | suffix before | answer[i] × suffix | suffix after (× nums[i]) |
|---|---|---|---|
| 3 | 1 | 6 × 1 = **6** | 4 |
| 2 | 4 | 2 × 4 = **8** | 12 |
| 1 | 12 | 1 × 12 = **12** | 24 |
| 0 | 24 | 1 × 24 = **24** | 24 |

Result `[24, 12, 8, 6]` ✅

**Order matters in the right pass:** multiply `answer[i]` by `suffix` *before* folding `nums[i]` into `suffix`. Otherwise you'd include `nums[i]` in its own product.

In [3]:
cases = [
    ([1, 2, 3, 4], [24, 12, 8, 6]),
    ([-1, 1, 0, -3, 3], [0, 0, 9, 0, 0]),
    ([0, 0], [0, 0]),           # two zeros -> all zero
    ([0, 5], [5, 0]),           # one zero
    ([2, 3], [3, 2]),
    ([-2, -3, 4], [-12, -8, 6]),
]
for nums, expected in cases:
    for Impl in (SolutionBrute, SolutionPrefixSuffix, Solution):
        assert Impl().productExceptSelf(nums) == expected, (Impl.__name__, nums)
    print(nums, "->", expected)

import random
for _ in range(500):
    nums = [random.randint(-3, 3) for _ in range(random.randint(2, 8))]
    assert Solution().productExceptSelf(nums) == SolutionBrute().productExceptSelf(nums)
print("All tests passed ✅")

[1, 2, 3, 4] -> [24, 12, 8, 6]
[-1, 1, 0, -3, 3] -> [0, 0, 9, 0, 0]
[0, 0] -> [0, 0]
[0, 5] -> [5, 0]
[2, 3] -> [3, 2]
[-2, -3, 4] -> [-12, -8, 6]
All tests passed ✅


### Step 4 · Wrap up

| | Time | Extra space | Why |
|---|---|---|---|
| Brute force | O(n²) | O(1) | Recomputes overlapping products |
| Prefix + suffix arrays | O(n) | O(n) | Two helper arrays |
| **Prefix in output + suffix scalar** | **O(n)** | **O(1)** | Output array reused; one running variable |

**Complexity, explained:** there are two separate linear passes, each doing O(1) work per index, so the total is O(2n) = O(n). Extra memory is one integer (`suffix`). The `answer` array is required output, so by the problem's rules it doesn't count.

**Edge cases:** one zero → only that index is non-zero. Two or more zeros → all zeros. Negatives → signs come out right automatically. n = 2 → swap.

**Follow-ups:**
- *If division were allowed?* → Count zeros: 0 zeros → `total // nums[i]`; 1 zero → only the zero index gets the product of the non-zeros; 2+ zeros → all zero. Still O(n), but there are more cases to get wrong.
- *Same pattern elsewhere:* prefix/suffix scans also solve Trapping Rain Water (Q16), Candy (Q15), and range-sum queries.

🗣️ **30-second recap:** *"Brute force multiplies everything else for each index, which is O(n²). Each answer is just left-product times right-product. I build the left products in the output array in one pass, then sweep from the right with a running suffix product and multiply it in. No division, zeros handled naturally, O(n) time and O(1) extra space."*